In [ ]:
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt
import pystac_client
import planetary_computer
import odc.stac
import rasterio.features
from sklearn.metrics import f1_score, accuracy_score

# ==============================================================================
# 1. HELPER FUNCTIONS FOR IN-MEMORY STAC & RASTER REDUCTION
# ==============================================================================

def get_utm_crs(gdf: gpd.GeoDataFrame) -> str:
    """Determine the local UTM EPSG code from a GeoDataFrame centroid."""
    centroid = gdf.to_crs("EPSG:4326").union_all().centroid
    lon, lat = centroid.x, centroid.y
    zone = int((lon + 180) // 6) + 1
    epsg = 32600 + zone if lat >= 0 else 32700 + zone
    return f"EPSG:{epsg}"


def compute_event_ndvi_in_memory(
    row: pd.Series,
    catalog_client: pystac_client.Client,
    buffer_days: int = 10,
    max_cloud_cover: int = 80,
    resolution: float = 10.0
) -> float:
    """Stream optical bands for a single event directly into memory and return

    the two-stage reduced NDVI: temporal min first, then spatial mean.
    """
    geom_4326 = row.geometry
    
    # 1. Date window with buffer
    start_dt = (pd.to_datetime(row["start_date"]) - pd.Timedelta(days=buffer_days)).strftime("%Y-%m-%d")
    end_dt = (pd.to_datetime(row["end_date"]) + pd.Timedelta(days=buffer_days)).strftime("%Y-%m-%d")
    
    # 2. STAC Search directly in memory
    search = catalog_client.search(
        collections=["sentinel-2-l2a"],
        intersects=geom_4326.__geo_interface__,
        datetime=f"{start_dt}/{end_dt}",
        query={"eo:cloud_cover": {"lte": max_cloud_cover}},
        limit=100
    )
    items = list(search.items())
    
    if not items:
        return np.nan

    # Sign asset URLs for Planetary Computer Azure SAS authentication
    items = [planetary_computer.sign(item) for item in items]
    
    # 3. Project polygon to local UTM for accurate 10m spatial rasterization
    event_gdf = gpd.GeoDataFrame([row], crs="EPSG:4326")
    utm_crs = get_utm_crs(event_gdf)
    event_utm = event_gdf.to_crs(utm_crs)
    minx, miny, maxx, maxy = event_utm.total_bounds
    
    # 4. Stream Red (B04) and NIR (B08) directly into an xarray DataArray
    # (xarray chunks/streams directly from Azure Blob Storage COGs)
    ds = odc.stac.load(
        items,
        bands=["B04", "B08"],
        crs=utm_crs,
        resolution=resolution,
        x=(minx - 20, maxx + 20),
        y=(miny - 20, maxy + 20),
        chunks={}
    ).load() # Materialize arrays into RAM

    if ds.sizes.get("time", 0) == 0:
        return np.nan

    # 5. Vectorized NDVI Calculation across 3D Array (time, y, x)
    # Apply ESA baseline offset correction if needed (>4000 DN)
    b8 = ds["B08"].astype("float32")
    b4 = ds["B04"].astype("float32")
    
    ndvi_stack = (b8 - b4) / (b8 + b4 + 1e-6)
    
    # 6. Stage 1: Temporal Aggregation First (Temporal MIN per spatial pixel)
    ndvi_min_2d = ndvi_stack.min(dim="time")
    
    # 7. Rasterize Field Polygon to Mask Pixels Outside Boundary
    transform = ds.odc.transform
    height, width = ds.sizes["y"], ds.sizes["x"]
    
    mask = rasterio.features.rasterize(
        [(geom, 1) for geom in event_utm.geometry],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype="uint8"
    )
    
    # 8. Stage 2: Spatial Aggregation (Spatial MEAN across polygon mask)
    masked_ndvi = np.where(mask == 1, ndvi_min_2d.values, np.nan)
    spatial_mean_ndvi = np.nanmean(masked_ndvi)
    
    return float(spatial_mean_ndvi)


# ==============================================================================
# 2. MAIN BATCH EXECUTION PIPELINE
# ==============================================================================

# Input GeoDataFrame setup (replace 'gdf_tillage' with your GeoDataFrame)
gdf = gdf_tillage.copy().to_crs("EPSG:4326")
if "field_id" not in gdf.columns:
    gdf["field_id"] = [f"event_{i}" for i in range(len(gdf))]

# Connect to Planetary Computer STAC Catalog ONCE
catalog_client = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)

print(f"Starting in-memory processing for {len(gdf)} tillage events...")

# Run in-memory extraction over events
ndvi_results = []
for idx, row in gdf.iterrows():
    print(f"Processing Event {idx+1}/{len(gdf)}: {row['field_id']} ({row['till_type']})...", end="")
    try:
        ndvi_val = compute_event_ndvi_in_memory(row, catalog_client, buffer_days=10)
        ndvi_results.append(ndvi_val)
        print(f" Done -> NDVI: {ndvi_val:.4f}")
    except Exception as e:
        print(f" Failed -> Error: {e}")
        ndvi_results.append(np.nan)

gdf["summary_ndvi"] = ndvi_results

# Filter out failed/empty retrievals
analysis_df = gdf.dropna(subset=["summary_ndvi"]).copy()
